# Brute Force Detection & Correlation Engine

### Analyzes auth logs and correlates with port scan data

**Author:**  Raiza Rosas

**Date:** 2026-02-12

In [ ]:
import re
from datetime import datetime, timedelta
from collections import defaultdict
import json

class BruteForceDetector:
    def __init__(self, auth_log_file, scan_iocs_file=None):
        self.auth_log = auth_log_file
        self.scan_iocs = scan_iocs_file
        
        # Data structures
        self.failed_attempts = defaultdict(lambda: {
            'count': 0,
            'usernames': set(),
            'timestamps': [],
            'ports': []
        })
        
        self.successful_logins = []
        self.attack_timeline = []
        self.scan_data = None
        
        # Thresholds
        self.BRUTE_FORCE_THRESHOLD = 10  # 10+ failed attempts = suspicious
        self.TIME_WINDOW = 300  # 5 minutes (300 seconds)
        
    def load_scan_data(self):
        """Load port scan IOCs from Day 11"""
        if not self.scan_iocs:
            return
            
        try:
            with open(self.scan_iocs, 'r') as f:
                self.scan_data = json.load(f)
            print(f"[+] Loaded scan data from {self.scan_iocs}")
        except FileNotFoundError:
            print(f"[!] Scan IOC file not found: {self.scan_iocs}")
    
    def parse_auth_log(self):
        """Parse authentication logs"""
        print(f"\n[*] Parsing auth log: {self.auth_log}")
        
        # Regex patterns
        failed_pattern = r'(\w{3}\s+\d{1,2}\s+\d{2}:\d{2}:\d{2}).*Failed password for (?:invalid user )?(\w+) from ([\d.]+) port (\d+)'
        accepted_pattern = r'(\w{3}\s+\d{1,2}\s+\d{2}:\d{2}:\d{2}).*Accepted password for (\w+) from ([\d.]+) port (\d+)'
        
        with open(self.auth_log, 'r') as f:
            for line in f:
                # Failed attempts
                failed_match = re.search(failed_pattern, line)
                if failed_match:
                    timestamp_str, username, src_ip, port = failed_match.groups()
                    timestamp = self.parse_timestamp(timestamp_str)
                    
                    self.failed_attempts[src_ip]['count'] += 1
                    self.failed_attempts[src_ip]['usernames'].add(username)
                    self.failed_attempts[src_ip]['timestamps'].append(timestamp)
                    self.failed_attempts[src_ip]['ports'].append(int(port))
                
                # Successful logins
                accepted_match = re.search(accepted_pattern, line)
                if accepted_match:
                    timestamp_str, username, src_ip, port = accepted_match.groups()
                    timestamp = self.parse_timestamp(timestamp_str)
                    
                    self.successful_logins.append({
                        'timestamp': timestamp,
                        'username': username,
                        'src_ip': src_ip,
                        'port': int(port)
                    })
        
        print(f"[+] Parsed {sum(d['count'] for d in self.failed_attempts.values())} failed attempts")
        print(f"[+] Parsed {len(self.successful_logins)} successful logins")
    
    def parse_timestamp(self, timestamp_str):
        """Convert syslog timestamp to datetime"""
        # Format: "Feb 12 18:23:45"
        current_year = datetime.now().year
        timestamp_str_with_year = f"{timestamp_str} {current_year}"
        return datetime.strptime(timestamp_str_with_year, "%b %d %H:%M:%S %Y")
    
    def detect_brute_force(self):
        """Detect brute force patterns"""
        print("\n" + "="*70)
        print("BRUTE FORCE DETECTION RESULTS")
        print("="*70)
        
        for src_ip, data in self.failed_attempts.items():
            if data['count'] >= self.BRUTE_FORCE_THRESHOLD:
                # Calculate attack duration
                timestamps = sorted(data['timestamps'])
                duration = (timestamps[-1] - timestamps[0]).total_seconds()
                
                # Calculate rate
                rate = data['count'] / (duration / 60) if duration > 0 else data['count']
                
                # Severity scoring
                if data['count'] > 1000:
                    severity = "CRITICAL"
                elif data['count'] > 100:
                    severity = "HIGH"
                else:
                    severity = "MEDIUM"
                
                print(f"\n{severity} BRUTE FORCE DETECTED")
                print(f"  Source IP: {src_ip}")
                print(f"  ├─ Failed Attempts: {data['count']}")
                print(f"  ├─ Unique Usernames: {len(data['usernames'])}")
                print(f"  ├─ Usernames Tried: {', '.join(sorted(data['usernames']))}")
                print(f"  ├─ Attack Duration: {duration:.2f} seconds ({duration/60:.2f} minutes)")
                print(f"  ├─ Attack Rate: {rate:.2f} attempts/minute")
                print(f"  ├─ First Attempt: {timestamps[0].strftime('%Y-%m-%d %H:%M:%S')}")
                print(f"  └─ Last Attempt: {timestamps[-1].strftime('%Y-%m-%d %H:%M:%S')}")
                
                # Check for successful login after brute force
                self.check_breach(src_ip, timestamps[-1])
    
    def check_breach(self, attacker_ip, last_failed_attempt):
        """Check if brute force resulted in successful login"""
        for login in self.successful_logins:
            if login['src_ip'] == attacker_ip:
                time_diff = (login['timestamp'] - last_failed_attempt).total_seconds()
                
                if time_diff > 0 and time_diff < 600:  # Within 10 minutes
                    print(f"\nBREACH CONFIRMED ")
                    print(f"  ├─ Successful Login: {login['timestamp'].strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"  ├─ Compromised Account: {login['username']}")
                    print(f"  ├─ Time After Last Failed Attempt: {time_diff:.2f} seconds")
                    print(f"  └─ Recommendation: IMMEDIATE PASSWORD RESET + INVESTIGATE ACTIVITY")
    
    def correlate_with_scan(self):
        """Correlate brute force with port scan data"""
        if not self.scan_data:
            print("\n[!] No scan data available for correlation")
            return
        
        print("\n" + "="*70)
        print("ATTACK CHAIN CORRELATION (Scan → Brute Force)")
        print("="*70)
        
        # Extract scan timestamps from Day 11 IOCs
        for malicious_ip_data in self.scan_data.get('malicious_ips', []):
            scan_ip = malicious_ip_data['ip_address']
            
            # Check if this IP also performed brute force
            if scan_ip in self.failed_attempts:
                print(f"\nCORRELATION FOUND: {scan_ip}")
                print(f"  Phase 1: Port Scan (T1046)")
                print(f"    ├─ Ports Scanned: {malicious_ip_data['ports_scanned']}")
                print(f"    ├─ Scan Types: {', '.join(malicious_ip_data['scan_types'])}")
                print(f"    └─ Last Seen: {malicious_ip_data['last_seen']}")
                
                print(f"\n  Phase 2: Brute Force Attack (T1110)")
                print(f"    ├─ Failed Attempts: {self.failed_attempts[scan_ip]['count']}")
                print(f"    ├─ Targeted Usernames: {len(self.failed_attempts[scan_ip]['usernames'])}")
                print(f"    └─ Service: SSH (port 22)")
                
                # Calculate time between scan and brute force
                scan_time = datetime.fromisoformat(malicious_ip_data['last_seen'])
                brute_force_start = min(self.failed_attempts[scan_ip]['timestamps'])
                time_gap = (brute_force_start - scan_time).total_seconds()
                
                print(f"\n Time Gap Between Phases: {time_gap/60:.2f} minutes")
                
                if time_gap < 3600:  # Less than 1 hour
                    print(f"RAPID PROGRESSION - Automated attack likely")
    
    def generate_iocs(self):
        """Generate IOCs for SIEM ingestion"""
        iocs = {
            'metadata': {
                'timestamp': datetime.now().isoformat(),
                'log_analyzed': self.auth_log,
                'detection_type': 'brute_force',
                'analyst': '[Your Name]'
            },
            'brute_force_attacks': []
        }
        
        for src_ip, data in self.failed_attempts.items():
            if data['count'] >= self.BRUTE_FORCE_THRESHOLD:
                timestamps = sorted(data['timestamps'])
                duration = (timestamps[-1] - timestamps[0]).total_seconds()
                
                # Check if breach occurred
                breach = False
                compromised_account = None
                for login in self.successful_logins:
                    if login['src_ip'] == src_ip:
                        breach = True
                        compromised_account = login['username']
                
                iocs['brute_force_attacks'].append({
                    'attacker_ip': src_ip,
                    'failed_attempts': data['count'],
                    'usernames_tried': list(data['usernames']),
                    'attack_duration_seconds': duration,
                    'first_attempt': timestamps[0].isoformat(),
                    'last_attempt': timestamps[-1].isoformat(),
                    'breach_occurred': breach,
                    'compromised_account': compromised_account,
                    'severity': 'CRITICAL' if data['count'] > 1000 else 'HIGH' if data['count'] > 100 else 'MEDIUM',
                    'mitre_attack': ['T1110.001', 'T1110.003'] if breach else ['T1110.001']
                })
        
        # Save to file
        output_file = 'iocs_brute_force.json'
        with open(output_file, 'w') as f:
            json.dump(iocs, f, indent=4)
        
        print(f"\n{'='*70}")
        print(f"[+] IOCs saved to: {output_file}")
        print(f"[+] Total brute force attacks identified: {len(iocs['brute_force_attacks'])}")
        print("="*70 + "\n")
    
    def run_analysis(self):
        """Main analysis workflow"""
        self.load_scan_data()
        self.parse_auth_log()
        self.detect_brute_force()
        self.correlate_with_scan()
        self.generate_iocs()

if __name__ == "__main__":
    import sys
    
    if len(sys.argv) < 2:
        print("Usage: python3 correlation_engine.py <auth_log_file> [scan_iocs.json]")
        sys.exit(1)
    
    auth_log = sys.argv[1]
    scan_iocs = sys.argv[2] if len(sys.argv) > 2 else None
    
    detector = BruteForceDetector(auth_log, scan_iocs)
    detector.run_analysis()